# Multimodal VQA - Kaggle Training
Mount a Kaggle Dataset containing VQA v2 annotations and COCO 2014 `train2014/` and `val2014/` directories before running.

In [ ]:
import os
import subprocess
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/multimodal-vqa-data/vqa")
WORK_ROOT = Path("/kaggle/working/multimodal-vqa")
REPO_ROOT = Path("/kaggle/working/multimodal-vqa-repo")
CONFIG_PATH = "configs/kaggle_finetune.yaml"  # or configs/baseline_frozen.yaml
RUN_NAME = "finetune"
TOTAL_EPOCHS = 12
GIT_REF = "main"  # replace with a release tag or commit for strict reproducibility
CHECKPOINT_DIR = WORK_ROOT / RUN_NAME
ANSWER_VOCAB = WORK_ROOT / "answer_vocab.json"
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/dongtingshuo/multimodal-vqa.git", str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
subprocess.run(["git", "checkout", GIT_REF], check=True)
subprocess.run(["python", "-m", "pip", "install", "-r", "requirements.txt"], check=True)

In [ ]:
subprocess.run(
    [
        "python",
        "scripts/validate_vqa_data.py",
        "--root",
        str(DATA_ROOT),
        "--sample-images",
        "20",
    ],
    check=True,
)

## One-epoch calibration
Run this once to measure epoch time. The generated `latest.pt` can be resumed by the next cell.

In [ ]:
subprocess.run(
    [
        "python",
        "train.py",
        "--config",
        CONFIG_PATH,
        "--device",
        "cuda",
        "--data-root",
        str(DATA_ROOT),
        "--answer-vocab-path",
        str(ANSWER_VOCAB),
        "--checkpoint-dir",
        str(CHECKPOINT_DIR),
        "--epochs",
        "1",
    ],
    check=True,
)

## Full training or resume
Re-running this cell resumes from `latest.pt`. Save the Notebook version so `/kaggle/working` artifacts remain available.

In [ ]:
command = [
    "python",
    "train.py",
    "--config",
    CONFIG_PATH,
    "--device",
    "cuda",
    "--data-root",
    str(DATA_ROOT),
    "--answer-vocab-path",
    str(ANSWER_VOCAB),
    "--checkpoint-dir",
    str(CHECKPOINT_DIR),
    "--epochs",
    str(TOTAL_EPOCHS),
]
latest = CHECKPOINT_DIR / "latest.pt"
if latest.exists():
    command.extend(["--resume", str(latest)])
subprocess.run(command, check=True)

In [ ]:
predictions = CHECKPOINT_DIR / "val_predictions.json"
subprocess.run(
    [
        "python",
        "evaluate.py",
        "--config",
        CONFIG_PATH,
        "--checkpoint",
        str(CHECKPOINT_DIR / "best.pt"),
        "--device",
        "cuda",
        "--data-root",
        str(DATA_ROOT),
        "--predictions-output",
        str(predictions),
    ],
    check=True,
)
subprocess.run(
    ["tar", "-czf", str(WORK_ROOT / f"{RUN_NAME}_artifacts.tar.gz"), "-C", str(CHECKPOINT_DIR), "."], check=True
)